# Checkpoints


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Tn.source(source_str)
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r})
    .sink(sink_str)
)

built_tn = Tn.build(tn)



In [ ]:
import random

class OrderGenerator:
    def __init__(self):
        self.order_id_int = 0
        self.customer_id_int = 0
        #
        self.ts_int = 0
        self.ts_step_int = 1

    def generate(self):
        m = {
            "key": self.order_id_int,
            "value": {"id": self.order_id_int,
                      "product_id": random.randint(0, 100 - 1),
                      "customer_id": random.randint(0, 10 - 1),
                      "ts": self.ts_int},
        }
        #
        self.order_id_int += 1
        #
        self.ts_int += self.ts_step_int
        #
        return m

#

gen = OrderGenerator()
for _ in range(3):
    print(gen.generate())


In [ ]:
built_tn.reset()
gen = OrderGenerator()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [ ]:
import cloudpickle

#

gen = OrderGenerator()


#

built_tn.reset()
source_m_list = []
sink_m_list = []
for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

x = cloudpickle.dumps(built_tn._evaluator)

# print(built_tn.latest())

built_tn.reset()

# print(built_tn.latest())

y = cloudpickle.loads(x)

built_tn._evaluator = y

# print(built_tn.latest())

#

for _ in range(10):
    m = gen.generate()
    source_m_list += [m]
    built_tn.push(source_str, [m])
    sink_m_list += built_tn.latest()[sink_str]

#

source_key_int_value_dict_dict = {}
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["key"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")


In [ ]:
import sys
sys.path.insert(1, "..")

import kafi.streams.streams
import importlib
importlib.reload(kafi.streams.streams)

from kafi.kafka.cluster.cluster import Cluster
from kafi.streams.streams import Streams

c = Cluster({"kafka": {"bootstrap.servers": "localhost:9092"}})

source_str = "orders"
sink_str = "orders_aggregated"

tn = (
    Streams.source(c, source_str)
    
    .map(lambda r: r["value"])
    .group_by_agg(lambda r: r["customer_id"],
                  lambda r: r,
                  lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                    "order_ids": sorted(agg_r["order_ids"] + [r["id"]]),
                                    "product_ids": sorted(agg_r["product_ids"] + [r["product_id"]])},
                  {"orders": 0, "order_ids": [], "product_ids": []},
                  lambda by, agg_r: {"customer_id": by,
                                     "orders": agg_r["orders"],
                                     "order_ids": agg_r["order_ids"],
                                     "product_ids": agg_r["product_ids"]})
    .map(lambda r: {"key": r["customer_id"],
                    "value": r}).peek("sink")
    .sink(c, sink_str)
)

built_tn = Streams.build(tn)



In [ ]:
from kafi.helpers import get_millis

built_tn.reset()

checkpoint_str = "checkpoint"
g = f"group_{get_millis()}"

c.recreate(source_str)
c.recreate(sink_str)
c.recreate(checkpoint_str)

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, checkpoint_interval=0.01, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)
gen = OrderGenerator()

#

for _ in range(10):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



In [ ]:
await stop_fun()
await Streams.tasks()

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(10):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()



In [ ]:
await stop_fun()
await Streams.tasks()

In [ ]:
built_tn.reset()

print(c.l(source_str))
print(c.l(sink_str))
print(c.l(checkpoint_str))

#

stop_fun = Streams.start_streams_task(built_tn, checkpoint_storage=c, checkpoint_topic=checkpoint_str, group=g)
# stop_fun = Streams.start_streams_task(built_tn, group=g)

#

pr = c.producer(source_str)

#

for _ in range(10):
    m = gen.generate()
    pr.produce_list([m])

#

pr.close()


In [ ]:
await stop_fun()
await Streams.tasks()

In [ ]:
source_key_int_value_dict_dict = {}
source_m_list = c.cat(source_str)
for m in source_m_list:
    order_id_int = m["value"]["id"]
    product_id_int = m["value"]["product_id"]
    customer_id_int = m["value"]["customer_id"]
    #
    agg_orders_int = source_key_int_value_dict_dict.get(customer_id_int, {}).get("orders", 0)
    agg_order_id_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("order_ids", [])
    agg_product_ids_int_list = source_key_int_value_dict_dict.get(customer_id_int, {}).get("product_ids", [])
    #
    source_key_int_value_dict_dict[customer_id_int] = {"customer_id": customer_id_int,
                                                       "orders": agg_orders_int + 1,
                                                       "order_ids": sorted(agg_order_id_int_list + [order_id_int]),
                                                       "product_ids": sorted(agg_product_ids_int_list + [product_id_int])}

#

sink_key_int_value_dict_dict = {}
sink_m_list = c.cat(sink_str)
for m in sink_m_list:
    sink_key_int_value_dict_dict[m["value"]["customer_id"]] = m["value"]

#

print(source_key_int_value_dict_dict)
print(sink_key_int_value_dict_dict)

if source_key_int_value_dict_dict != sink_key_int_value_dict_dict:
    raise Exception("Test failed.")

#

print("Test successful.")
